# Download BGT data from PDOK

This notebook assists in downloading the required BGT data which is needed for the post-processing steps in `detection_postprocessing.ipynb`. The source is the [PDOK API - swagger UI](https://api.pdok.nl/lv/bgt/download/v1_0/ui/#/).

In [ ]:
from datetime import datetime

from aerial_image_detection import settings
from aerial_image_detection.utils.city_area_handler import CityAreaHandler
from aerial_image_detection.utils.pdok_downloader import PDOKDownloader

In [ ]:
# We first define the area for which we want to download the BGT data, based on the relevant city area.
# NOTE: the CityAreaHandler only works for the municipality of Amsterdam

buurten_gdf = CityAreaHandler().get_city_area_gdf()

# Get the target area from the config.yml
target_area = settings["inference"]["target_area"]
(scale, name) = next(iter(target_area.items()))
download_area = buurten_gdf[buurten_gdf[f"{scale}_naam"] == name].union_all()

# Specify the required features (grouped by purpose)
download_features = ["wegdeel"]
extract_bgt_functions = {
    "parkeervlakken": ["parkeervlak"],
    "ontheffinggebied": ["voetpad", "fietspad", "voetpad op trap", "voetgangersgebied"],
}

# Specify download location and file format
timestamp = datetime.now().strftime(format="%Y-%m-%dT%H-%M-%S")
download_folder = f"../datasets/experiments/parkeren/bgt_{timestamp}"
extension = ".gpkg"

In [ ]:
# Download the required data
pdok_downloader = PDOKDownloader()

files = pdok_downloader.download_features_for_area(
    features=download_features,
    area = download_area,
    download_dir=download_folder,
    extract_bgt_functions=extract_bgt_functions,
    suffix=name.lower(),
    extension=extension,
)

for key, value in files.items():
    print(f"{key} stored in: {value}")